# Gradient Boosting Regressor — Solution

**Difficulty**: 🔴 Hard

---

### Problem Statement

Gradient-boosted decision trees (XGBoost, LightGBM, CatBoost) are still the strongest models for
tabular data: ranking and ads features, fraud, forecasting, and many Kaggle wins. In ML-breadth
interviews you're expected to explain where the *gradient* in gradient boosting comes from and to
implement it.

Boosting builds an additive model $F_M(x) = F_0 + \nu\sum_{m=1}^{M} h_m(x)$ one small tree at a
time. Each new tree $h_m$ is fit to the **negative gradient** of the loss with respect to the
current predictions. For the squared loss $L = \tfrac12\,(y - F)^2$ that negative gradient is the
**residual** $y - F(x)$. The learning rate $\nu$ ("shrinkage") scales every step.

You will write the boosting loop **and** the weak learner: a CART regression tree whose splits
minimize squared error and whose leaves predict the mean residual.

---

### Requirements

1. **`GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, min_samples_split=2)`**
2. **`fit(X, y)`**: `X` is `(N, D)` float, `y` is `(N,)` float. Returns `self`. Sets
   **`init_`** (a float) to $F_0 = \operatorname{mean}(y)$.
3. **Boosting loop** for $m = 1, \dots,$ `n_estimators`:
   - residuals $r_i = y_i - F_{m-1}(x_i)$;
   - fit a regression tree $h_m$ to $(X, r)$;
   - $F_m = F_{m-1} + \nu\,h_m$.
4. **Regression tree** (your own; a helper class is fine):
   - binary splits `x[f] <= t`, with candidate thresholds at the **midpoints between consecutive
     distinct sorted values** of the feature among the node's samples;
   - choose the split minimizing $\mathrm{SSE}_L + \mathrm{SSE}_R$ (variance reduction). Split only
     if that is **strictly lower** than the node's own SSE;
   - a leaf predicts the **mean** target (residual) of its samples;
   - `max_depth` (root = depth 0, `None` = unlimited) and `min_samples_split` (a node with fewer
     samples is a leaf), as in CART.
5. **`predict(X)`** → `(M,)`: $F_0 + \nu \sum_m h_m(X)$.
6. **`staged_predict(X)`**: a list or generator of `n_estimators` tensors of shape `(M,)`. Entry
   `i` is the prediction using the first `i + 1` trees, *not* including the bare $F_0$. Each entry
   must be its own tensor, so `list(staged_predict(X))` is correct.

---

### Constraints

- ✅ PyTorch tensor ops. Loops over boosting stages and tree nodes are fine.
- ✅ Vectorized split search: one sort + cumulative sums per node, not a loop over thresholds.
- ❌ No scikit-learn, XGBoost or LightGBM.

---

<details>
  <summary>💡 Hint</summary>

  - Which constant $c$ minimizes $\sum_i (y_i - c)^2$? That's your $F_0$.
  - $\mathrm{SSE} = \sum r^2 - S^2/n$, where $S = \sum r$. Sort a feature once: for the cut after
    sorted position $i$, $S_L$ is a prefix sum of the sorted residuals and $S_R = S - S_L$.
  - Minimizing $\mathrm{SSE}_L + \mathrm{SSE}_R$ is the same as maximizing $S_L^2/n_L + S_R^2/n_R$.
    Mask cuts between equal feature values.
  - Why is the mean the right leaf value for squared loss? What would it be for absolute loss?
  - With squared loss and $0 < \nu \le 1$, should the training MSE ever go up from one stage to the next?

</details>

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
# A noisy sine wave: easy to see, impossible for a single shallow tree.
torch.manual_seed(0)
X = torch.rand(300, 1) * 2 * math.pi                 # (300, 1)
y = torch.sin(X[:, 0]) + 0.1 * torch.randn(300)      # (300,), noise variance 0.01
X_train, y_train, X_test, y_test = X[:200], y[:200], X[200:], y[200:]
print("X_train", tuple(X_train.shape), "y_train", tuple(y_train.shape))

In [ ]:
class RegressionTree:
    """CART regression tree: splits minimize the children's summed squared error,
    leaves predict the mean target of their samples."""

    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def fit(self, X, r):
        """X: (N, D) features, r: (N,) targets (the residuals, when boosting)."""
        feature, threshold, left, right, value = [], [], [], [], []
        self.depth_ = 0

        def new_node(idx, depth):
            value.append(r[idx].mean())                       # leaf = mean target
            feature.append(-1); threshold.append(0.0); left.append(-1); right.append(-1)
            self.depth_ = max(self.depth_, depth)
            return len(value) - 1

        all_idx = torch.arange(X.shape[0])
        stack = [(new_node(all_idx, 0), all_idx, 0)]
        while stack:
            node, idx, depth = stack.pop()
            if self.max_depth is not None and depth >= self.max_depth:
                continue
            if len(idx) < max(self.min_samples_split, 2):
                continue
            split = _best_sse_split(X[idx], r[idx])
            if split is None:
                continue
            f, t = split
            go_left = X[idx, f] <= t
            feature[node], threshold[node] = f, t
            left[node] = new_node(idx[go_left], depth + 1)
            right[node] = new_node(idx[~go_left], depth + 1)
            stack.append((left[node], idx[go_left], depth + 1))
            stack.append((right[node], idx[~go_left], depth + 1))

        self.feature_ = torch.tensor(feature)
        self.threshold_ = torch.tensor(threshold, dtype=X.dtype)
        self.left_, self.right_ = torch.tensor(left), torch.tensor(right)
        self.value_ = torch.stack(value)
        return self

    def predict(self, X):
        """(M, D) -> (M,): route every row down the tree at once, one level per step."""
        rows = torch.arange(X.shape[0])
        node = torch.zeros(X.shape[0], dtype=torch.long)
        for _ in range(self.depth_):
            f = self.feature_[node]
            go_left = X[rows, f.clamp(min=0)] <= self.threshold_[node]
            node = torch.where(f < 0, node, torch.where(go_left, self.left_[node], self.right_[node]))
        return self.value_[node]


def _best_sse_split(X, r):
    """Best (feature, threshold) by variance reduction, or None if nothing helps.

    For a cut after sorted position i: SSE_L + SSE_R = sum(r^2) - (S_L^2/n_L + S_R^2/n_R),
    with S_L a prefix sum of the targets in sorted order. One sort + one cumsum scores
    every threshold of every feature.
    """
    n, D = X.shape
    xs, order = X.sort(dim=0)                                  # (n, D)
    r = r.double()                                             # cumsums in float64
    s_left = r[order].cumsum(dim=0)[:-1]                       # (n-1, D) target sum left of cut
    s_total = r.sum()
    n_left = torch.arange(1, n, dtype=torch.float64)[:, None]  # (n-1, 1)
    score = s_left ** 2 / n_left + (s_total - s_left) ** 2 / (n - n_left)
    score = score.masked_fill(xs[1:] == xs[:-1], float("-inf"))  # no cut between equal values
    best = int(score.argmax())
    i, f = divmod(best, D)
    gain = score[i, f] - s_total ** 2 / n                     # = SSE_parent - (SSE_L + SSE_R)
    if not gain > 1e-12 * (r ** 2).sum():                     # strict (relative) decrease
        return None
    return f, float((xs[i, f] + xs[i + 1, f]) / 2)            # midpoint threshold


class GradientBoostingRegressor:
    """Gradient-boosted regression trees for squared loss."""

    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, min_samples_split=2):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        """X: (N, D), y: (N,). Returns self."""
        self.init_ = float(y.mean())                          # F_0 = argmin_c sum (y - c)^2
        F = torch.full_like(y, self.init_)                    # current predictions F_m(x_i)
        self.trees_ = []
        for _ in range(self.n_estimators):
            residual = y - F                                  # = -dL/dF for L = (y - F)^2 / 2
            tree = RegressionTree(self.max_depth, self.min_samples_split).fit(X, residual)
            F = F + self.learning_rate * tree.predict(X)      # shrunken step
            self.trees_.append(tree)
        return self

    def staged_predict(self, X):
        """Yield the (M,) prediction after each of the n_estimators stages."""
        F = torch.full((X.shape[0],), self.init_, dtype=X.dtype)
        for tree in self.trees_:
            F = F + self.learning_rate * tree.predict(X)
            yield F

    def predict(self, X):
        """(M, D) -> (M,)."""
        F = torch.full((X.shape[0],), self.init_, dtype=X.dtype)
        for tree in self.trees_:
            F = F + self.learning_rate * tree.predict(X)
        return F

In [ ]:
# Test 1: one stage of depth 1 with learning_rate=1 == the best regression stump (brute force)
def brute_force_stump(X, y):
    y = y.double()
    best = (float("inf"), None, None)
    for f in range(X.shape[1]):
        vals = torch.unique(X[:, f])
        for t in ((vals[:-1] + vals[1:]) / 2).tolist():
            m = X[:, f] <= t
            sse = (((y[m] - y[m].mean()) ** 2).sum() + ((y[~m] - y[~m].mean()) ** 2).sum()).item()
            if sse < best[0]:
                best = (sse, f, t)
    return best

sse, f, t = brute_force_stump(X_train, y_train)
stump = GradientBoostingRegressor(n_estimators=1, learning_rate=1.0, max_depth=1).fit(X_train, y_train)
left = X_train[:, f] <= t
want = torch.where(left, y_train[left].mean(), y_train[~left].mean())
assert abs(stump.init_ - y_train.mean().item()) < 1e-6, "init_ must be mean(y)"
assert torch.allclose(stump.predict(X_train), want, atol=1e-5), "not the best SSE stump"
print(f"best stump: x[{f}] <= {t:.4f}, SSE {sse:.3f}")
print("Test 1 passed")

In [ ]:
# Test 2: training MSE never goes up across stages; the full model fits the sine
gbr = GradientBoostingRegressor().fit(X_train, y_train)
staged = list(gbr.staged_predict(X_train))
assert len(staged) == gbr.n_estimators, f"expected {gbr.n_estimators} stages, got {len(staged)}"
assert torch.allclose(staged[-1], gbr.predict(X_train), atol=1e-6)
mse = [((p - y_train) ** 2).mean().item() for p in staged]
assert all(b <= a * (1 + 1e-5) + 1e-7 for a, b in zip(mse[:-1], mse[1:])), "training MSE went up"
test_mse = ((gbr.predict(X_test) - y_test) ** 2).mean().item()
print("train MSE after 1, 10, 50, 100 stages:", [round(mse[i], 4) for i in (0, 9, 49, 99)])
print(f"test MSE {test_mse:.4f} (noise variance 0.01)")
assert mse[-1] < 0.015 and test_mse < 0.03
print("Test 2 passed")

In [ ]:
# Test 3: the learning rate scales the first stage's update exactly
a = GradientBoostingRegressor(n_estimators=1, learning_rate=1.0).fit(X_train, y_train)
b = GradientBoostingRegressor(n_estimators=1, learning_rate=0.1).fit(X_train, y_train)
base = y_train.mean()
assert torch.allclose(b.predict(X_test) - base, 0.1 * (a.predict(X_test) - base), atol=1e-6)
zero = GradientBoostingRegressor(n_estimators=5, learning_rate=0.0).fit(X_train, y_train)
assert torch.allclose(zero.predict(X_test), base.expand(len(X_test)))
print("Test 3 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("gradient-boosting", GradientBoostingRegressor=GradientBoostingRegressor)

In [ ]:
# Optional: shrinkage trades stages for generalization (test MSE after 10 / 50 / 200 stages)
for lr in (1.0, 0.3, 0.1, 0.03):
    model = GradientBoostingRegressor(n_estimators=200, learning_rate=lr).fit(X_train, y_train)
    test = [((p - y_test) ** 2).mean().item() for p in model.staged_predict(X_test)]
    print(f"lr={lr:<5} test MSE @10: {test[9]:.4f}  @50: {test[49]:.4f}  @200: {test[199]:.4f}"
          f"  (best {min(test):.4f} at stage {test.index(min(test)) + 1})")

## Why it works

**Boosting is gradient descent in function space.** We want to minimize
$\mathcal L(F) = \sum_i L(y_i, F(x_i))$ over functions $F$. Treat the vector of training predictions
$(F(x_1), \dots, F(x_N))$ as the parameters. Its gradient is $g_i = \partial L / \partial F(x_i)$.
We can't move $F$ by $-g$ directly, because that wouldn't generalize to new $x$. So we fit a tree
$h_m$ to $-g$ by least squares, i.e. project the step onto the space of trees, and take a step
$F \leftarrow F + \nu h_m$ (Friedman, 2001).

**Squared loss.** With $L = \tfrac12(y-F)^2$: $-g_i = y_i - F(x_i)$, the residual. So every stage
fits the current residuals. The leaf value "mean residual" is also the exact per-leaf line search
$\gamma = \arg\min_\gamma \sum_{i\in\text{leaf}} L(y_i, F(x_i) + \gamma)$. For other losses only the
pseudo-residuals and this line search change. $F_0 = \operatorname{mean}(y)$ is the best constant.

**Training MSE never increases** for $0 \le \nu \le 2$. In a leaf with residuals $r_i$ and mean
$\bar r$, $\sum_i (r_i - \nu\bar r)^2 = \sum_i (r_i - \bar r)^2 + n\bar r^2(1-\nu)^2 \le \sum_i r_i^2$.
The grader checks this along `staged_predict`. If you fit trees to $y$ instead of the
residuals, or forget to update the residuals, the training MSE starts rising within a few stages
and the check fails.

**Vectorized split search.** $\mathrm{SSE} = \sum r^2 - S^2/n$, so for a cut with prefix sum $S_L$:
$\mathrm{SSE}_L + \mathrm{SSE}_R = \sum r^2 - \big(S_L^2/n_L + S_R^2/n_R\big)$. Sort each feature
(`X.sort(dim=0)`), take `r[order].cumsum(0)`, and every cut of every feature is scored in one
`(n−1, D)` expression: $O(n\log n)$ per feature instead of $O(n^2)$. The gain
$S_L^2/n_L + S_R^2/n_R - S^2/n = \frac{n_L n_R}{n}(\bar r_L - \bar r_R)^2$ is positive iff the two
sides have different means. The reference computes it in float64, because a prefix sum followed
by a difference of squares cancels badly in float32.

**Exact properties the grader uses.** A single depth-1 stage with $\nu = 1$ is
$\bar y + \text{stump}(y - \bar y)$, the best regression stump on $y$ itself. With one stage, the
update is linear in $\nu$, because $h_1$ is fit to $y - \bar y$, which doesn't depend on $\nu$.

## Complexity

- **Fit:** $O(M \cdot \text{depth} \cdot D\,N\log N)$ for $M$ stages. The sort order of each feature
  never changes between stages (only the residuals do), so a production implementation sorts once
  up front, or bins features into histograms, and drops the $\log N$.
- **Predict:** $O(M \cdot \text{depth})$ per sample; every tree routes all rows at once.
- **Memory:** $O(M \cdot 2^{\text{depth}})$ for the trees.

## Common bugs

- **Fitting each tree to $y$** (or to $y - F_0$ every time) instead of the *current* residuals.
- **Shrinkage applied in only one place**: full steps in training but $\nu$ at prediction time, or the reverse.
- **Scaling $F_0$ by the learning rate.**
- **Median leaves.** That's the line search for *absolute* loss, not squared loss.
- **`staged_predict` that mutates one tensor in place** (`F += ...; yield F`). `list(...)` then holds
  $M$ references to the final prediction. Also: yielding $F_0$ as an extra first stage.
- The usual tree bugs: cutting between equal values, thresholds not at midpoints, depth off by one.

## Interview follow-ups

1. **Gradient boosting vs. random forests?** Random forests grow deep, decorrelated trees
   independently (bootstrap samples + random feature subsets) and average them. That cuts
   *variance*, parallelizes trivially, and more trees never hurt. Boosting grows shallow trees
   *sequentially*, each correcting the current residuals. That cuts *bias*, is usually more accurate
   once tuned, but can overfit with too many stages, so it needs early stopping on validation loss.
2. **What does shrinkage buy, and how does it interact with `n_estimators`?** Smaller $\nu$ takes
   smaller steps along the path of trees. You need roughly proportionally more stages, but
   generalization is better (the optional demo cell shows it). Tune $\nu$ and $M$ together with early
   stopping. Row subsampling per stage (stochastic gradient boosting) and column subsampling
   regularize further.
3. **What changes for other losses?** Only the pseudo-residuals and the leaf values. Absolute loss:
   $-g = \operatorname{sign}(y - F)$, leaf = median residual. Huber: a mix of the two. Log-loss for
   binary classification on logits: $-g = y - \sigma(F)$, leaf $\approx \sum r / \sum p(1-p)$ (a Newton step).
4. **What is XGBoost's second-order split gain?** XGBoost takes a second-order Taylor expansion of
   the loss, with per-sample $g_i$ and $h_i$, plus L2 regularization $\lambda$ on the leaf weights
   and a per-leaf penalty $\gamma$. The optimal leaf weight is $w^* = -G/(H+\lambda)$ and
   $$\text{gain} = \tfrac12\Big[\tfrac{G_L^2}{H_L+\lambda} + \tfrac{G_R^2}{H_R+\lambda} - \tfrac{(G_L+G_R)^2}{H_L+H_R+\lambda}\Big] - \gamma .$$
   For squared loss $h_i = 1$ and $g_i = -r_i$, so with $\lambda = \gamma = 0$ this is exactly half
   the variance-reduction score used here, and $w^*$ is the mean residual.
5. **How do XGBoost/LightGBM scale?** Histogram split finding: features are quantized into ≤ 256 bins
   once, gradient/hessian sums are accumulated per bin, and a child's histogram is the parent's
   minus its sibling's. LightGBM adds leaf-wise growth, GOSS (sampling by gradient magnitude)
   and exclusive feature bundling, and all of them have GPU kernels.
6. **How do you interpret a boosted model?** Gain-based feature importance is biased toward
   high-cardinality features. TreeSHAP computes exact Shapley values for tree ensembles in
   polynomial time, and partial-dependence / ICE plots show per-feature effects.